### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\rupam.pal\AppData\Local\Temp\ipykernel_16776\1573117932.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\rupam.pal\OneDrive - Euromonitor International\Personal Projects\RAG-Fundamentals\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read all the pdf's inside the directory - Data Ingestion (Document Structure)

In [2]:

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents



### Process all PDFs in the data directory


In [3]:
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Epicor_Complete_Revision_Guide.pdf
  ✓ Loaded 23 pages

Processing: epicor_interview_prep_rp.pdf
  ✓ Loaded 34 pages

Processing: Fundamentals_Prep_Rupam.pdf
  ✓ Loaded 10 pages

Total documents loaded: 67


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Qt 5.15.13', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-07-20T12:27:10+00:00', 'title': '', 'source': '..\\data\\pdf\\Epicor_Complete_Revision_Guide.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'Epicor_Complete_Revision_Guide.pdf', 'file_type': 'pdf'}, page_content='Epicor\tInterview\t—\tComplete\tRevision\tGuide\nProduct\tDeveloper\t(.NET,\tAngular,\tSQL)\t—\tRound\t1,\t21\tJuly\t2026\nEpicor\t—\tProduct\tDeveloper\t(.NET,\tAngular,\tSQL)\t—\tDeep\nPrep\tGuide\nRound\t1\t(Teams)\t—\t21\tJuly\t2026,\t12\tPM\nThis\tguide\tis\tbuilt\tdirectly\tagainst\tthe\tJD\tyou\tshared:\tC#,\t.NET\t8+,\tASP.NET\tCore,\tAngular\t15+,\tEF\tCore,\tSQL\tServer,\narchitecture\tpatterns,\ttesting\t(xUnit/Moq),\tAzure,\tDocker/K8s\tbasics,\tauth\t(OAuth2/JWT/OIDC),\tand\tAgile.\tGiven\tyou’re\t3\tyears\nin\tagainst\ta\t3–5\tyear\tband,\tthe\tinterviewer\twill\tlikely\tprobe\tdepth\tover\tbreadth\t—\texpect\t“why,”\tnot\tjust\t“what.”\tAns

### Text splitting get into chunks

In [5]:


def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 67 documents into 107 chunks

Example chunk:
Content: Epicor	Interview	—	Complete	Revision	Guide
Product	Developer	(.NET,	Angular,	SQL)	—	Round	1,	21	July	2026
Epicor	—	Product	Developer	(.NET,	Angular,	SQL)	—	Deep
Prep	Guide
Round	1	(Teams)	—	21	July	20...
Metadata: {'producer': 'Qt 5.15.13', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-07-20T12:27:10+00:00', 'title': '', 'source': '..\\data\\pdf\\Epicor_Complete_Revision_Guide.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'Epicor_Complete_Revision_Guide.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Qt 5.15.13', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-07-20T12:27:10+00:00', 'title': '', 'source': '..\\data\\pdf\\Epicor_Complete_Revision_Guide.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'Epicor_Complete_Revision_Guide.pdf', 'file_type': 'pdf'}, page_content='Epicor\tInterview\t—\tComplete\tRevision\tGuide\nProduct\tDeveloper\t(.NET,\tAngular,\tSQL)\t—\tRound\t1,\t21\tJuly\t2026\nEpicor\t—\tProduct\tDeveloper\t(.NET,\tAngular,\tSQL)\t—\tDeep\nPrep\tGuide\nRound\t1\t(Teams)\t—\t21\tJuly\t2026,\t12\tPM\nThis\tguide\tis\tbuilt\tdirectly\tagainst\tthe\tJD\tyou\tshared:\tC#,\t.NET\t8+,\tASP.NET\tCore,\tAngular\t15+,\tEF\tCore,\tSQL\tServer,\narchitecture\tpatterns,\ttesting\t(xUnit/Moq),\tAzure,\tDocker/K8s\tbasics,\tauth\t(OAuth2/JWT/OIDC),\tand\tAgile.\tGiven\tyou’re\t3\tyears\nin\tagainst\ta\t3–5\tyear\tband,\tthe\tinterviewer\twill\tlikely\tprobe\tdepth\tover\tbreadth\t—\texpect\t“why,”\tnot\tjust\t“what.”\tAns

### Embeddings 

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings # type: ignore


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9126.33it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\rupam.pal\AppData\Local\Temp\ipykernel_16776\2180998919.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vectorstore db

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
chunks


[Document(metadata={'producer': 'Qt 5.15.13', 'creator': 'wkhtmltopdf 0.12.6', 'creationdate': '2026-07-20T12:27:10+00:00', 'title': '', 'source': '..\\data\\pdf\\Epicor_Complete_Revision_Guide.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'Epicor_Complete_Revision_Guide.pdf', 'file_type': 'pdf'}, page_content='Epicor\tInterview\t—\tComplete\tRevision\tGuide\nProduct\tDeveloper\t(.NET,\tAngular,\tSQL)\t—\tRound\t1,\t21\tJuly\t2026\nEpicor\t—\tProduct\tDeveloper\t(.NET,\tAngular,\tSQL)\t—\tDeep\nPrep\tGuide\nRound\t1\t(Teams)\t—\t21\tJuly\t2026,\t12\tPM\nThis\tguide\tis\tbuilt\tdirectly\tagainst\tthe\tJD\tyou\tshared:\tC#,\t.NET\t8+,\tASP.NET\tCore,\tAngular\t15+,\tEF\tCore,\tSQL\tServer,\narchitecture\tpatterns,\ttesting\t(xUnit/Moq),\tAzure,\tDocker/K8s\tbasics,\tauth\t(OAuth2/JWT/OIDC),\tand\tAgile.\tGiven\tyou’re\t3\tyears\nin\tagainst\ta\t3–5\tyear\tband,\tthe\tinterviewer\twill\tlikely\tprobe\tdepth\tover\tbreadth\t—\texpect\t“why,”\tnot\tjust\t“what.”\tAns

In [15]:
### convert the texts to embeddings
texts = [doc.page_content for doc in chunks]

#### generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

#### add the documents and embeddings to the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 107 texts...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.53it/s]


Generated embeddings with shape: (107, 384)
Adding 107 documents to vector store...
Successfully added 107 documents to vector store
Total documents in collection: 107
